# 🧬 BioKG Tutorial Part 2: BioBridge & Multimodal Embeddings

In Part 1, we formatted 8.1M graphs into a Parquet format. Our Graph looks like this:
- `[Head Index: 423] -> [Relation: 12] -> [Tail Index: 5040]`

If we feed this directly into a deep learning model, it initializes Nodes `423` and `5040` with **Random Vectors (Noise)**. The model has to learn the entire biological universe from scratch just by observing connections. This takes massive amounts of Compute.

### 💡 The Solution: Transfer Learning via BioBridge
What if we pre-loaded the nodes with vectors that *already* represent biological reality? 

**BioBridge** (NeurIPS 2024 paper from Stanford/Tencent) calculated world-class abstractions for entities:
1. **Proteins**: Fed amino-acid sequences into **ESM-2b** ➡ Outputs a **2560-dimension** vector.
2. **Drugs**: Fed chemical SMILES into an encoder ➡ Outputs a **512-dimension** vector.
3. **Diseases**: Fed Medical text into **PubMedBERT** ➡ Outputs a **768-dimension** vector.

Let's load these and project them into PyTorch natively.

## 📥 Step 1: Download Embedding Dictionary Dumps
Make sure you ran the data script: `python data/download_biobridge.py`

We will load the massive `.pkl` dictionaries matching PrimeKG node indexes directly to their multi-modal arrays.

In [ ]:
import sys
sys.path.append('.')

!pip install loguru -q
import torch
import json

from models.biobridge_encoder import BioBridgeProjector

# Load mapping stats
with open('data/processed/stats.json') as f:
    stats = json.load(f)
max_idx = stats['max_node_index']

print(f"📊 Need to prepare space for {max_idx:,} biological entities in our PyTorch model.")

## 🔀 Step 2: Linear Dimensionality Mapping
Our Deep Learning KGE (Knowledge Graph Embedding) model expects all entities to have the **exact same dimensions** to perform calculations. 
How do we compare a 2560-dim Protein with a 512-dim Drug?

**Answer: Linear Projections!**
We instantiate a PyTorch `nn.Module` that accepts the raw tensors, and feeds them through standard Linear (Dense) layers, projecting them all to a unified space: **768 dimensions**.


In [ ]:
print("Initializing BioBridge Projector...")
# This takes ~15 seconds to parse and mount gigabytes of numpy arrays into Ram.
projector = BioBridgeProjector(max_node_index=max_idx, target_dim=768)

print("\nMounting Pre-Calculated ESM-2b/PubMed Data into Neural buffers...")
projector.load_pretrained_mappings()

print("\n✅ Projector is armed and ready.")

## 🧪 Step 3: Simulating the Forward Pass
Let's see the Projector in action! We will ask it to retrieve the embedding for some random node indexes. The Projector will figure out internally if it's a Drug, Disease, or Protein, grab the raw array, run it through the correct linear projection, and output clean 768-D vectors.

In [ ]:
# Simulate querying a batch of nodes from our RotatE algorithm
query_nodes = torch.tensor([1050, 4233, 9084, 120000])

unified_embeddings = projector(query_nodes)

print(f"🔹 Input Nodes Shape: {query_nodes.shape}")
print(f"🔹 Output Embeddings Shape: {unified_embeddings.detach().numpy().shape}")
print("\nWe queried 4 entities, and received 4 vectors consisting of exactly 768 dimensions.")

### 🎉 Summary of Part 2
Instead of blindly training raw integers against other integers, we now have an Object-Oriented module that safely caches millions of biomedical features. 

Whenever RotatE attempts to calculate a Graph relationship score, it will fetch these dynamically bridged dimensions. This acts as a massive "Bootstrap" constraint, ensuring drugs that are structurally similar in SMILES-space will start out close together in Graph Space!

➡ **Proceed to Tutorial 3**: We will initiate the PyTorch Lightning trainer across epochs and optimize the Graph!